In [ ]:
# Импорты
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import log_loss

# 1. Загрузка данных
data = pd.read_csv('data/gbm-data.csv')
X = data.iloc[:, 1:].values
y = data.iloc[:, 0].values

# 2. Разбивка
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.8, random_state=241
)

# Сигмоида с защитой от overflow
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -250, 250)))

# 3. Обучение GB и построение графиков
learning_rates = [1, 0.5, 0.3, 0.2, 0.1]
results = {}

plt.figure(figsize=(15, 10))

for i, lr in enumerate(learning_rates, 1):
    print(f"Обработка learning_rate = {lr}")
    
    gbm = GradientBoostingClassifier(
        n_estimators=250,
        learning_rate=lr,
        random_state=241,
        verbose=False
    )
    gbm.fit(X_train, y_train)
    
    train_losses = []
    test_losses = []
    
    for y_pred_train, y_pred_test in zip(
        gbm.staged_decision_function(X_train),
        gbm.staged_decision_function(X_test)
    ):
        y_proba_train = sigmoid(y_pred_train)
        y_proba_test = sigmoid(y_pred_test)
        train_losses.append(log_loss(y_train, y_proba_train))
        test_losses.append(log_loss(y_test, y_proba_test))
    
    results[lr] = {
        'train': train_losses,
        'test': test_losses,
        'min_test_loss': min(test_losses),
        'best_iter': test_losses.index(min(test_losses)) + 1
    }
    
    # График
    plt.subplot(2, 3, i)
    plt.plot(train_losses, label='train', color='blue')
    plt.plot(test_losses, label='test', color='orange')
    plt.title(f'learning_rate = {lr}')
    plt.xlabel('Итерация')
    plt.ylabel('log-loss')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()

# 4. Анализ overfitting для lr=0.2
lr02 = results[0.2]
test_losses_02 = lr02['test']
min_idx = test_losses_02.index(lr02['min_test_loss'])

# Проверяем, растёт ли loss после минимума
after_min = test_losses_02[min_idx:]
is_overfitting = any(after_min[i] < after_min[i+1] for i in range(len(after_min) - 1))

overfitting_str = "overfitting" if is_overfitting else "underfitting"
print(f"Поведение при lr=0.2: {overfitting_str}")
print(f"Минимальный log-loss на тесте: {lr02['min_test_loss']:.5f}")
print(f"Достигается на итерации: {lr02['best_iter']}")

# 5. Random Forest
best_iter = lr02['best_iter']
rf = RandomForestClassifier(n_estimators=best_iter, random_state=241)
rf.fit(X_train, y_train)

y_proba_rf = rf.predict_proba(X_test)  # уже вероятности, сигмоида НЕ нужна
rf_loss = log_loss(y_test, y_proba_rf)

print(f"Log-loss случайного леса: {rf_loss:.5f}")